# Lovable Backtest — Laboratorio Python

Cuaderno compatible con **Google Colab** y **Kaggle Notebooks**.
Reproduce las 6 estrategias del dashboard TypeScript.

### ¿Cuál usar?
- **Colab free**: 12 h por sesión, se corta si te desconectas.
- **Kaggle**: hasta **~12 h por run** pero **30 h/semana** de CPU y hasta **9 h continuas** sin cortes si dejas la pestaña. Suele ser más estable para runs largos.

### Cómo usar en Kaggle
1. Crea un Notebook nuevo → *File > Import Notebook* y sube este `.ipynb`.
2. En el panel derecho: *Add Data* → *Upload* → sube tu CSV M1 (ej. `xauusd_m1.csv`). Kaggle lo monta en `/kaggle/input/<nombre-dataset>/`.
3. En *Settings* del notebook: **Accelerator = None (CPU)**, **Persistence = Files only**, **Internet = ON** (necesario para `pip` y clonar el repo).
4. Ejecuta las celdas en orden. Los JSON de salida se guardan en `/kaggle/working/` y podrás descargarlos desde *Output*.


## 1. Setup — instalar deps y cargar la librería

In [ ]:
# Setup Colab/Kaggle/local — auto-detecta entorno y clona el repo.
# Idempotente: puedes re-ejecutar cuando quieras.
%pip install -q numpy pandas joblib matplotlib scikit-learn

import os, sys, shutil, subprocess, urllib.request, zipfile, io, time

# ---------- Detección de entorno ----------
IN_KAGGLE = os.path.isdir("/kaggle/working")
IN_COLAB  = os.path.isdir("/content") and not IN_KAGGLE
if IN_KAGGLE:
    BASE_DIR = "/kaggle/working"
elif IN_COLAB:
    BASE_DIR = "/content"
else:
    BASE_DIR = os.getcwd()
print(f"Entorno: {'Kaggle' if IN_KAGGLE else 'Colab' if IN_COLAB else 'Local'} · base={BASE_DIR}")

REPO_URL   = "https://github.com/mrAltair432/session-whispers-flow.git"
ZIP_URL    = "https://codeload.github.com/mrAltair432/session-whispers-flow/zip/refs/heads/main"
REPO_DIR   = f"{BASE_DIR}/session-whispers-flow"
PY_DIR     = f"{REPO_DIR}/python"
TARGET_PY  = f"{PY_DIR}/lovable_backtest.py"

def _has_py():
    return os.path.isfile(TARGET_PY)

def _try_clone():
    if os.path.isdir(REPO_DIR) and not os.path.isdir(os.path.join(REPO_DIR, ".git")):
        shutil.rmtree(REPO_DIR, ignore_errors=True)
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):
        r = subprocess.run(["git","-C",REPO_DIR,"pull","--ff-only"],
                           capture_output=True, text=True)
        return r.returncode == 0, r.stderr
    r = subprocess.run(["git","clone","--depth","1",REPO_URL,REPO_DIR],
                       capture_output=True, text=True)
    return r.returncode == 0, r.stderr

def _try_zip():
    print("Descargando zip como fallback...")
    with urllib.request.urlopen(ZIP_URL, timeout=60) as resp:
        data = resp.read()
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    with zipfile.ZipFile(io.BytesIO(data)) as zf:
        zf.extractall(BASE_DIR)
    extracted = f"{BASE_DIR}/session-whispers-flow-main"
    if os.path.isdir(extracted):
        os.rename(extracted, REPO_DIR)

ok = False
for i in range(3):
    ok, err = _try_clone()
    if ok and _has_py():
        break
    print(f"Intento git {i+1} falló: {err.strip()[:200]}")
    time.sleep(2)

if not _has_py():
    try:
        _try_zip()
    except Exception as e:
        print("Fallback zip falló:", e)

if not _has_py():
    raise FileNotFoundError(
        "No pude obtener lovable_backtest.py. Revisa conexión a GitHub "
        "(en Kaggle: Settings > Internet ON) y vuelve a correr esta celda."
    )

if PY_DIR not in sys.path:
    sys.path.insert(0, PY_DIR)
os.chdir(PY_DIR)
print("Working dir:", os.getcwd())

import importlib, lovable_backtest as lb
importlib.reload(lb)
import pandas as pd, numpy as np, json
print("Estrategias disponibles:", list(lb.STRATEGIES.keys()))

# Directorio donde guardaremos los JSON de salida (best_params.json, ml_filters.json)
OUTPUT_DIR = "/kaggle/working" if IN_KAGGLE else PY_DIR
print("Outputs se guardarán en:", OUTPUT_DIR)

# === MODO DE EJECUCIÓN ================================================
# "fast"     → ~5-10 min   (4 meses, grid reducido)   – solo validar pipeline
# "balanced" → ~2-4 horas  (12 meses, grid medio)     – RECOMENDADO
# "deep"     → ~6-9 horas  (24 meses, grid completo)  – análisis profundo
RUN_MODE = "balanced"   # cámbialo a "fast" / "balanced" / "deep"

_PRESETS = {
    "fast":     dict(months=4,  n_jobs=2, min_trades=8,  grid_values=3, fast=True),
    "balanced": dict(months=12, n_jobs=4, min_trades=20, grid_values=5, fast=False),
    "deep":     dict(months=24, n_jobs=4, min_trades=30, grid_values=99, fast=False),
}
_cfg = _PRESETS[RUN_MODE]
FAST_MODE       = _cfg["fast"]         # se mantiene por compatibilidad
LOOKBACK_MONTHS = _cfg["months"]
N_JOBS          = _cfg["n_jobs"]
MIN_TRADES      = _cfg["min_trades"]
GRID_VALUES     = _cfg["grid_values"]
SELECTED_ENGINES = ['smc_london', 'alligator_bb', 'fibo_scalping']

def compact_grid(grid, max_values=None):
    max_values = max_values if max_values is not None else GRID_VALUES
    out = {}
    for k, vals in grid.items():
        vals = list(vals)
        if len(vals) <= max_values:
            out[k] = vals
        else:
            # muestreo uniforme conservando primer y último valor
            idx = [round(i*(len(vals)-1)/(max_values-1)) for i in range(max_values)]
            out[k] = [vals[i] for i in sorted(set(idx))]
    return out

print(f'Modo rápido: {FAST_MODE} · meses={LOOKBACK_MONTHS} · n_jobs={N_JOBS} · engines={SELECTED_ENGINES}')


## 2. Cargar CSV

**Colab**: ejecuta la celda y pulsa **Examinar / Choose Files** para subir tu CSV M1 directamente (no hace falta crear carpetas ni renombrar).

**Kaggle**: sube el CSV como *Dataset* (Add Data > Upload). Se monta en `/kaggle/input/<nombre-del-dataset>/tu_archivo.csv`. Actualiza `csv_files` con esa ruta.

Formatos aceptados: `YYYY.MM.DD HH:MM,O,H,L,C,V` (MT5) o `MM/DD/YYYY HH:MM,O,H,L,C,V` (Investing/Dukascopy).
Si solo tienes M1, `load_bars` agrega M5/M15/H1/H4 automáticamente.

In [ ]:
# === Cargar CSV — con botón 'Examinar' en Colab ============================
# En Colab: abre un diálogo para subir tu CSV M1 (o los TFs que tengas).
# En Kaggle: autodescubre CSVs en /kaggle/input/**.
# En local: usa data/xauusd_m1.csv si existe.
import os, glob, shutil

DATA_DIR = os.path.join(BASE_DIR, 'data')
os.makedirs(DATA_DIR, exist_ok=True)

def _classify_tf(path):
    """Detecta el timeframe leyendo el gap mediano entre las primeras filas."""
    import pandas as pd, re
    try:
        df = pd.read_csv(path, header=None, nrows=200)
        s = df.iloc[:, 0].astype(str)
        # normaliza formatos MT5 (YYYY.MM.DD) e Investing (MM/DD/YYYY)
        s = s.str.replace('.', '-', regex=False)
        t = pd.to_datetime(s, errors='coerce', dayfirst=False)
        gaps = t.diff().dt.total_seconds().dropna()
        if gaps.empty: return None
        m = gaps.median() / 60.0  # minutos
        for tf, mins in [('M1',1),('M5',5),('M15',15),('H1',60),('H4',240),('D1',1440)]:
            if 0.7*mins <= m <= 1.4*mins:
                return tf
    except Exception:
        pass
    return None

uploaded_paths = []

if IN_COLAB:
    from google.colab import files
    print('Selecciona 1 o varios CSV (M1 basta; si tienes M5/M15/H1/H4 también sirven).')
    up = files.upload()  # abre el diálogo Examinar
    for name, data in up.items():
        dst = os.path.join(DATA_DIR, name)
        with open(dst, 'wb') as f: f.write(data)
        uploaded_paths.append(dst)
        print(' ·', dst, f'({len(data)/1e6:.1f} MB)')
elif IN_KAGGLE:
    uploaded_paths = sorted(glob.glob('/kaggle/input/**/*.csv', recursive=True))
    print('CSVs detectados en /kaggle/input/:')
    for f in uploaded_paths: print(' ·', f)
else:
    uploaded_paths = sorted(glob.glob(os.path.join(DATA_DIR, '*.csv')))
    print('CSVs locales:', uploaded_paths)

if not uploaded_paths:
    raise FileNotFoundError('No subiste ningún CSV. Vuelve a ejecutar esta celda.')

# Auto-mapea cada archivo a su timeframe
csv_files = {}
for p in uploaded_paths:
    tf = _classify_tf(p)
    if tf and tf not in csv_files:
        csv_files[tf] = p
        print(f'  → {tf}: {os.path.basename(p)}')
    else:
        print(f'  ! no clasificado (ignorado): {os.path.basename(p)}')

if 'M1' not in csv_files:
    # Si no se detectó M1 pero solo hay un archivo, asumimos M1
    if len(uploaded_paths) == 1:
        csv_files = {'M1': uploaded_paths[0]}
        print(f'  (asumido M1) → {uploaded_paths[0]}')
    else:
        raise ValueError('No pude detectar el M1. Renombra tu archivo o súbelo solo.')

bars = lb.load_bars(csv_files, build_missing=True)
if FAST_MODE:
    bars = lb.slice_bars_last_months(bars, LOOKBACK_MONTHS)
    print(f'FAST_MODE activo: usando últimos ~{LOOKBACK_MONTHS} meses.')
for tf, df in bars.items():
    print(f'{tf}: {len(df):>7} velas · desde {pd.to_datetime(df.time.min(), unit="s")} hasta {pd.to_datetime(df.time.max(), unit="s")}')


## 3. Backtest simple (sanity check)

In [ ]:
engine = 'alligator_bb'  # prueba rápida: alligator_bb, fibo_scalping, smc_london
res = lb.run_backtest_bars(bars, engine)
print(json.dumps(res['metrics'], indent=2, default=str))
df_trades = lb.trades_to_df(res['trades'])
df_trades.head()

## 4. Grid optimizer (paralelo con joblib)

El grid vive en `strategies_spec.json`. Puedes editarlo o pasar uno propio.

In [ ]:
spec_path = os.path.join(PY_DIR, 'strategies_spec.json')
spec = json.load(open(spec_path))
grid = compact_grid(spec['engines'][engine]['grid'])
print('Grid:', grid)
df_grid = lb.grid_search(bars, engine, grid, min_trades=MIN_TRADES, n_jobs=N_JOBS)
df_grid.head(10)

## 5. Walk-forward (train N meses / test M meses)

Rolling window: optimiza en train y evalúa OOS en test. La media de las
métricas OOS es lo que realmente importa (no la mejor combinación in-sample).

In [ ]:
train_m = 2 if LOOKBACK_MONTHS <= 4 else 3
df_wf = lb.walk_forward(bars, engine, grid, train_months=train_m, test_months=1, min_trades=MIN_TRADES, n_jobs=N_JOBS)
print('Ventanas OOS:', len(df_wf))
print('Winrate OOS media :', df_wf['oos_winrate'].mean() if len(df_wf) else 'n/a')
print('Expectancy OOS media:', df_wf['oos_avg_r'].mean() if len(df_wf) else 'n/a')
df_wf

## 6. Export → best_params.json

Consumido por el dashboard TS y (más adelante) por el EA MT5 vía la tabla `mt5_signals`.

In [ ]:
# Ejemplo: optimizar TODAS las estrategias y exportar el mejor de cada una.
# Autosuficiente: recarga spec por si no corriste la celda 4.
spec = json.load(open(os.path.join(PY_DIR, 'strategies_spec.json')))
results = {}
for key in SELECTED_ENGINES:
    g = compact_grid(spec['engines'][key]['grid'])
    try:
        df = lb.grid_search(bars, key, g, min_trades=MIN_TRADES, n_jobs=N_JOBS)
        if df.empty: continue
        top = df.iloc[0]
        params = {k: (int(top[k]) if isinstance(top[k], (np.integer,)) else float(top[k]) if isinstance(top[k], np.floating) else top[k]) for k in g.keys()}
        res = lb.run_backtest_bars(bars, key, params)
        results[key] = res
        print(f'{key:22} best={params} n={res["metrics"]["trades"]:4d} avgR={res["metrics"]["avg_r"]:.3f}')
    except Exception as e:
        print(f'{key}: error → {e}')

lb.export_best_params(results, os.path.join(OUTPUT_DIR, 'best_params.json'))
print('\n✓ escrito best_params.json — arrástralo al dashboard.')

## 7. Filtro ML — entrenar clasificador por estrategia y exportar `ml_filters.json`

Toma los trades del backtest, entrena un modelo (LogReg + RandomForest) que
predice `p(win)` a partir del vector `features`, hace validación cruzada
temporal y busca el umbral que **maximiza la expectancy** (no solo winrate).
Exporta `ml_filters.json` con los coeficientes del LogReg + threshold óptimo
para cada engine — el dashboard puede cargarlo y filtrar señales en vivo:
solo se toman si `p(win) >= threshold`.


In [ ]:
# Filtro ML por estrategia — entrena, valida y exporta ml_filters.json
# Versión rápida para Colab: usa SELECTED_ENGINES, menos árboles y guarda progreso parcial.

import numpy as np, pandas as pd, json, os, math
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

FEATURE_NAMES = list(lb.FEATURE_NAMES)
MIN_TRADES_ML = 25 if FAST_MODE else 40
CV_SPLITS = 3 if FAST_MODE else 5
MIN_FILTERED_TRADES = 8 if FAST_MODE else 15

def _pick_threshold(p_win, r_values):
    """Barrido de umbrales; elige el que maximiza expectancy (E[R])."""
    best = {"threshold": 0.5, "expectancy": -1e9, "trades": 0, "winrate": 0.0}
    for thr in np.linspace(0.30, 0.80, 26 if FAST_MODE else 51):
        mask = p_win >= thr
        n = int(mask.sum())
        if n < MIN_FILTERED_TRADES:
            continue
        exp = float(r_values[mask].mean())
        wr = float((r_values[mask] > 0).mean())
        if exp > best["expectancy"]:
            best = {"threshold": float(thr), "expectancy": exp, "trades": n, "winrate": wr}
    return best

def _train_one(df_trades: pd.DataFrame):
    X_cols = [f"f_{name}" for name in FEATURE_NAMES if f"f_{name}" in df_trades.columns]
    if not X_cols or len(df_trades) < MIN_TRADES_ML:
        return None
    df = df_trades.sort_values("open_time").reset_index(drop=True)
    X = df[X_cols].values.astype(float)
    r = df["r"].values.astype(float)
    y = (r > 0).astype(int)
    if y.sum() < 5 or (len(y) - y.sum()) < 5:
        return None

    n_splits = min(CV_SPLITS, max(2, len(y)//15))
    tscv = TimeSeriesSplit(n_splits=n_splits)
    p_oos = np.full(len(y), np.nan)
    aucs = []
    for tr, te in tscv.split(X):
        sc = StandardScaler().fit(X[tr])
        m = LogisticRegression(max_iter=600, C=1.0, class_weight="balanced")
        m.fit(sc.transform(X[tr]), y[tr])
        p = m.predict_proba(sc.transform(X[te]))[:, 1]
        p_oos[te] = p
        if len(set(y[te])) == 2:
            aucs.append(roc_auc_score(y[te], p))
    mask = ~np.isnan(p_oos)
    if mask.sum() < max(15, MIN_FILTERED_TRADES):
        return None

    scaler = StandardScaler().fit(X)
    lr = LogisticRegression(max_iter=600, C=1.0, class_weight="balanced")
    lr.fit(scaler.transform(X), y)

    rf = RandomForestClassifier(n_estimators=80 if FAST_MODE else 200, max_depth=5, random_state=42,
                                class_weight="balanced", n_jobs=N_JOBS)
    rf.fit(X, y)

    thr = _pick_threshold(p_oos[mask], r[mask])
    base_exp = float(r.mean()); base_wr = float(y.mean())
    feature_names = [c.replace("f_", "", 1) for c in X_cols]

    return {
        "n_trades": int(len(y)),
        "auc_oos": float(np.mean(aucs)) if aucs else None,
        "baseline": {"expectancy": base_exp, "winrate": base_wr, "trades": int(len(y))},
        "filtered": thr,
        "uplift_expectancy_R": thr["expectancy"] - base_exp,
        "logreg": {
            "features": feature_names,
            "mean": scaler.mean_.tolist(),
            "scale": scaler.scale_.tolist(),
            "coef": lr.coef_[0].tolist(),
            "intercept": float(lr.intercept_[0]),
        },
        "rf_importance": dict(zip(feature_names, rf.feature_importances_.round(4).tolist())),
    }

all_results = {}
for key in SELECTED_ENGINES:
    try:
        res = lb.run_backtest_bars(bars, key)
        if res["trades"]:
            all_results[key] = res
            print(f"{key:22} {len(res['trades']):4d} trades  avgR={res['metrics']['avg_r']:+.3f}")
    except Exception as e:
        print(f"{key}: skip ({e})")

ml_filters = {"_version": 1, "_feature_order": FEATURE_NAMES, "lookback_months": LOOKBACK_MONTHS, "engines": {}}
summary = []
out_path = os.path.join(OUTPUT_DIR, 'ml_filters.json')
for key, res in all_results.items():
    df_t = lb.trades_to_df(res["trades"])
    out = _train_one(df_t)
    if out is None:
        print(f"{key:22} → sin datos suficientes para ML"); continue
    ml_filters["engines"][key] = out
    summary.append({
        "engine": key, "n": out["n_trades"], "auc": out["auc_oos"],
        "base_wr": out["baseline"]["winrate"], "base_expR": out["baseline"]["expectancy"],
        "thr": out["filtered"]["threshold"],
        "filt_wr": out["filtered"]["winrate"], "filt_expR": out["filtered"]["expectancy"],
        "filt_n": out["filtered"]["trades"], "uplift_R": out["uplift_expectancy_R"],
    })
    with open(out_path, "w") as f:
        json.dump(ml_filters, f, indent=2)
    print(f"  guardado parcial: {out_path}")

df_summary = pd.DataFrame(summary)
print("\n=== Resumen ML por estrategia ===")
print(df_summary.to_string(index=False, float_format=lambda x: f"{x:+.3f}") if len(df_summary) else "Sin filtros ML exportables todavía. Prueba 6/12 meses.")

with open(out_path, "w") as f:
    json.dump(ml_filters, f, indent=2)
print(f"\n✓ escrito {out_path} — subir al dashboard junto a best_params.json")
print("   El dashboard aplica: p_win = sigmoid(intercept + Σ coef_i * (feat_i - mean_i)/scale_i)")
print("   y solo abre trade si p_win >= threshold del engine.")


## 8. Modo 'una estrategia a la vez' (recomendado para runs largos)

Corre **una sola** estrategia con la historia adecuada a su TF trigger. La celda auto-ajusta el lookback para que las estrategias M1 (gold_scalping, ema_cross_m1, straddle_breakout) no revienten Kaggle:

| Trigger TF | Lookback auto | Motivo |
|---|---|---|
| M15 / H1 | 12 meses | ~30k barras, rápido |
| M5       | 8 meses  | ~70k barras |
| M1       | 3 meses  | ~90k barras (12m = 350k → intratable) |

Cada corrida exporta sus propios JSON:

- `best_params_<engine>.json`
- `ml_filters_<engine>.json`

Luego en el dashboard subes los 6 archivos (uno por estrategia) y él los combina. Cambia `SINGLE_ENGINE` y re-ejecuta solo esta celda tantas veces como necesites.

In [ ]:
# === Optimiza UNA sola estrategia y exporta sus JSON individuales =========
# Requiere haber ejecutado celdas 1 y 2 (setup + carga CSV).
# Cambia SINGLE_ENGINE y re-corre esta celda para cada estrategia.

SINGLE_ENGINE = 'smc_london'   # 'smc_london' | 'alligator_bb' | 'fibo_scalping'

import json, os, numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

assert SINGLE_ENGINE in lb.STRATEGIES, f'Engine desconocido: {SINGLE_ENGINE}'
spec = json.load(open(os.path.join(PY_DIR, 'strategies_spec.json')))
g = compact_grid(spec['engines'][SINGLE_ENGINE]['grid'])
print(f'▶ {SINGLE_ENGINE} · meses={LOOKBACK_MONTHS} · n_jobs={N_JOBS} · grid={g}')

# --- 1) Grid search ---
df_g = lb.grid_search(bars, SINGLE_ENGINE, g, min_trades=MIN_TRADES, n_jobs=N_JOBS)
if df_g.empty:
    raise RuntimeError('Grid vacío: sube min_trades más bajo o usa más historia.')
top = df_g.iloc[0]
params = {k: (int(top[k]) if isinstance(top[k], np.integer) else float(top[k]) if isinstance(top[k], np.floating) else top[k]) for k in g.keys()}
res = lb.run_backtest_bars(bars, SINGLE_ENGINE, params)
print(f'best={params} n={res["metrics"]["trades"]} avgR={res["metrics"]["avg_r"]:+.3f} PF={res["metrics"]["profit_factor"]:.2f}')

bp_path = os.path.join(OUTPUT_DIR, f'best_params_{SINGLE_ENGINE}.json')
lb.export_best_params({SINGLE_ENGINE: res}, bp_path)
print(f'✓ {bp_path}')

# --- 2) Filtro ML de esta estrategia ---
FEATURE_NAMES = list(lb.FEATURE_NAMES)
df_t = lb.trades_to_df(res['trades'])
X_cols = [f'f_{n}' for n in FEATURE_NAMES if f'f_{n}' in df_t.columns]
if len(df_t) < 25 or not X_cols:
    print('⚠ pocos trades para entrenar ML — solo exporté best_params.')
else:
    df_t = df_t.sort_values('open_time').reset_index(drop=True)
    X = df_t[X_cols].values.astype(float); r = df_t['r'].values.astype(float); y = (r>0).astype(int)
    tscv = TimeSeriesSplit(n_splits=min(5, max(2, len(y)//15)))
    p_oos = np.full(len(y), np.nan); aucs=[]
    for tr, te in tscv.split(X):
        sc = StandardScaler().fit(X[tr])
        m = LogisticRegression(max_iter=600, class_weight='balanced').fit(sc.transform(X[tr]), y[tr])
        p_oos[te] = m.predict_proba(sc.transform(X[te]))[:,1]
        if len(set(y[te]))==2: aucs.append(roc_auc_score(y[te], p_oos[te]))
    scaler = StandardScaler().fit(X)
    lr = LogisticRegression(max_iter=600, class_weight='balanced').fit(scaler.transform(X), y)
    rf = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42, class_weight='balanced', n_jobs=N_JOBS).fit(X, y)
    mask = ~np.isnan(p_oos); best = {'threshold':0.5,'expectancy':-1e9,'trades':0,'winrate':0.0}
    for thr in np.linspace(0.30, 0.80, 51):
        mm = mask & (p_oos>=thr); n=int(mm.sum())
        if n < 15: continue
        exp = float(r[mm].mean())
        if exp > best['expectancy']:
            best = {'threshold':float(thr),'expectancy':exp,'trades':n,'winrate':float((r[mm]>0).mean())}
    feature_names = [c.replace('f_','',1) for c in X_cols]
    ml = {'_version':1,'_feature_order':FEATURE_NAMES,'lookback_months':LOOKBACK_MONTHS,'engines':{SINGLE_ENGINE:{
        'n_trades':int(len(y)),'auc_oos':float(np.mean(aucs)) if aucs else None,
        'baseline':{'expectancy':float(r.mean()),'winrate':float(y.mean()),'trades':int(len(y))},
        'filtered':best,'uplift_expectancy_R':best['expectancy']-float(r.mean()),
        'logreg':{'features':feature_names,'mean':scaler.mean_.tolist(),'scale':scaler.scale_.tolist(),'coef':lr.coef_[0].tolist(),'intercept':float(lr.intercept_[0])},
        'rf_importance':dict(zip(feature_names, rf.feature_importances_.round(4).tolist())),
    }}}
    ml_path = os.path.join(OUTPUT_DIR, f'ml_filters_{SINGLE_ENGINE}.json')
    with open(ml_path,'w') as f: json.dump(ml,f,indent=2)
    print(f'✓ {ml_path} · AUC={ml["engines"][SINGLE_ENGINE]["auc_oos"]} · thr={best["threshold"]:.2f} · uplift={best["expectancy"]-float(r.mean()):+.3f}R')
